# 03 · Robustez, Grad-CAM y corrección del falso positivo

## Por qué existe este cuaderno
Las curvas de entrenamiento **no detectan** cambios de dominio: la validación sale de las mismas fotos. Lo único que
los detecta son pruebas que cambian **un factor a la vez** y miden cuánto se mueve la respuesta del modelo.

## Hipótesis a contrastar con evidencia
> *"El falso positivo de la pared lisa con la mano se debe a que el modelo no aprendió con variación de brillo."*

El código de `construir_modelo` aplica la aumentación (incluida `RandomBrightness`) **antes** de `preprocess_input`,
así que la aumentación sí estuvo activa. Esta hipótesis se decide con la **prueba del atajo del brillo** (paso 2), no con
discusión: si casi ninguna imagen cambia de clase al oscurecerla, el brillo no es la causa.

In [8]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, datos, evaluar, entrenar, modelo, gradcam, ood, mosaico, predecir, fotos_propias, inclinacion

from tensorflow import keras
NOMBRE, ARQ = "v2_mobilenet", "mobilenetv2"
m = keras.models.load_model(f"models/{NOMBRE}/modelo.keras")
inf = modelo.modelo_inferencia(m, ARQ)
predecir_fn = entrenar.predictor(inf)
te = pd.read_csv("results/particion_prueba.csv")
va = pd.read_csv("results/particion_validacion.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto: /content/drive/MyDrive/proyecto_nuevo
Surface Crack: /content/datos
  Positive: 20000 imágenes
  Negative: 20000 imágenes


## Paso 0 · Control de integridad: ¿la prueba está limpia?
**Qué hace:** compara la partición de prueba actual con las particiones **con las que este modelo realmente se entrenó**
(las guarda `models/<nombre>/particion_*.csv`). Cuenta cuántas imágenes y cuántos grupos de la prueba el modelo ya vio.

**Por qué:** si el cuaderno 01 se vuelve a ejecutar DESPUÉS de entrenar (porque cambiaron las carpetas de fotos, por ejemplo),
`results/particion_prueba.csv` cambia, pero el modelo sigue siendo el viejo. Parte de la "prueba" nueva podría haber sido
entrenamiento, y el resultado saldría inflado sin ningún aviso.

**Qué hace si encuentra solapamiento:** descarta de la prueba lo que el modelo ya vio y sigue con la parte limpia.
Esa versión se guarda en `results/particion_prueba_verificada.csv`, que es la que usa el cuaderno 05.

In [9]:
tr_modelo = pd.read_csv(f"models/{NOMBRE}/particion_entrenamiento.csv")
va_modelo = pd.read_csv(f"models/{NOMBRE}/particion_validacion.csv")
solapa_img = te["ruta"].isin(set(tr_modelo["ruta"]) | set(va_modelo["ruta"]))
solapa_grp = te["grupo"].isin(set(tr_modelo["grupo"]) | set(va_modelo["grupo"]))
print(f"Prueba actual: {len(te):,} imágenes · {te['grupo'].nunique()} grupos")
print(f"  imágenes que el modelo YA VIO al entrenar o validar: {int(solapa_img.sum()):,}")
print(f"  imágenes cuyo GRUPO ya vio:                          {int(solapa_grp.sum()):,}")
if solapa_img.any() or solapa_grp.any():
    te = te[~solapa_img & ~solapa_grp].reset_index(drop=True)
    print(f"\n⚠ LA PRUEBA ESTABA CONTAMINADA. Se descartó lo ya visto. Prueba limpia: {len(te):,} imágenes.")
    print("  No uses los resultados de antes: repetí el Paso 1 con esta prueba limpia.")
else:
    print("\n✓ La prueba está limpia: ninguna imagen ni grupo coincide con el entrenamiento del modelo.")
te.to_csv("results/particion_prueba_verificada.csv", index=False)
print("Composición de la prueba verificada:", te["etiqueta"].value_counts().to_dict(), "| fuentes:", te["fuente"].value_counts().to_dict())

Prueba actual: 6,052 imágenes · 138 grupos
  imágenes que el modelo YA VIO al entrenar o validar: 0
  imágenes cuyo GRUPO ya vio:                          0

✓ La prueba está limpia: ninguna imagen ni grupo coincide con el entrenamiento del modelo.
Composición de la prueba verificada: {0: 3124, 1: 2928} | fuentes: {'surface_crack': 6052}


## Paso 1 · Resultado en PRUEBA (se mira UNA vez)
Es el resultado que se informa. Con división por grupos, **se espera que sea menor** que el 99,93 % anterior: esa
diferencia es la fuga que se eliminó, no un empeoramiento.

In [ ]:
prob_te = entrenar.predecir_rutas(predecir_fn, te["ruta"].tolist())
r = evaluar.metricas(te["etiqueta"].to_numpy(), prob_te)
lo, hi = evaluar.intervalo_wilson(r["fn"], r["fn"] + r["tp"])
print(r["matriz"], f"\nexactitud {r['exactitud']:.4f} · recall {r['recall']:.4f} · precisión {r['precision']:.4f}")
print(f"tasa de falsos negativos {100*r['fn']/(r['fn']+r['tp']):.2f} % (IC95 {100*lo:.2f}–{100*hi:.2f} %)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.2, 4))
ax.imshow(r["matriz"], cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, r["matriz"][i][j], ha="center", va="center", fontsize=14)
ax.set_xticks([0, 1], ["sin grieta", "con grieta"]); ax.set_yticks([0, 1], ["sin grieta", "con grieta"])
ax.set_xlabel("predicho"); ax.set_ylabel("real"); ax.set_title(f"Matriz de confusión (prueba, n={len(te):,})")
fig.tight_layout(); fig.savefig("results/matriz_confusion.png", dpi=150)
print("F1 =", round(r["f1"], 4), "| los falsos negativos (abajo a la izquierda) son las grietas NO detectadas")

## Paso 2 · Prueba del atajo del brillo
**Qué hace:** toma imágenes **sin grieta** y las **oscurece** (−18, −30, −51 niveles) y cuenta cuántas pasan a *con
grieta*; y al revés, aclara las **con grieta**. Solo cambia el brillo: misma textura, mismo concreto.

* 18 = la diferencia real entre clases · 51 = el máximo de la aumentación.
* **≈ 0 % cambia → el modelo NO usa el brillo.** Hasta ~5 % es tolerable.
* **Mucho % → el modelo usa el brillo como atajo.**

Se probó con modelos falsos (`tests/test_evaluar.py`): detecta a uno que usa el brillo y NO acusa a uno que mira la textura.

In [ ]:
n = 300
neg = entrenar.cargar_imagenes(te[te.etiqueta == 0]["ruta"].tolist()[:n])
pos = entrenar.cargar_imagenes(te[te.etiqueta == 1]["ruta"].tolist()[:n])
t_brillo = evaluar.prueba_atajo_brillo(predecir_fn, neg, pos, deltas=(18, 30, 51))
display(t_brillo)
print(evaluar.veredicto_atajo(t_brillo))
t_brillo.to_csv("results/prueba_atajo_brillo.csv", index=False)

## Paso 3 · Robustez: giro, lejanía, perspectiva, desenfoque, ruido
**Qué hace:** aplica cada transformación con varias intensidades y mide **% que cambia de clase**, exactitud y recall.

* **Giro hasta 54°:** lo que `RandomRotation(0.15)` cubre *en teoría*; aquí se mide *en la práctica*.
* **Lejanía:** simula una foto tomada de lejos (achicar y volver). Con 0,25 una grieta de 4 px queda en 1 px.
  Si el recall cae acá, **la solución es el mosaico** (paso 5).
* **Perspectiva:** foto en ángulo. Afecta más a la medición del ancho que a la clasificación.

In [15]:
mezcla = pd.concat([te[te.etiqueta == 0].head(150), te[te.etiqueta == 1].head(150)])
imgs = entrenar.cargar_imagenes(mezcla["ruta"].tolist())
t_rob = evaluar.prueba_robustez(predecir_fn, imgs, mezcla["etiqueta"].to_numpy())
display(t_rob)
t_rob.to_csv("results/prueba_robustez.csv", index=False)

,prueba,parametro,cambian_de_clase_%,exactitud_%,recall_%
0,giro_grados,5.00,0.0,100.0,100.0
1,giro_grados,15.00,0.0,100.0,100.0
2,giro_grados,30.00,0.3,99.7,99.3
3,giro_grados,45.00,0.3,99.7,99.3
4,giro_grados,54.00,0.7,99.3,98.7
5,giro_grados,90.00,0.3,99.7,99.3
6,lejania_escala,0.75,0.0,100.0,100.0
7,lejania_escala,0.50,0.0,100.0,100.0
8,lejania_escala,0.35,0.0,100.0,100.0
9,lejania_escala,0.25,1.0,99.0,98.0


## Paso 4 · Grad-CAM: ¿en qué se fija el modelo?
**Qué hace:** pinta sobre la foto las zonas que más empujaron la decisión hacia *grieta*.

Subí a `data/problemas/` tu foto de la **pared lisa con la mano** y otras fallas.

| Si marca… | Significa |
|---|---|
| bordes y sombras de los dedos | confunde líneas oscuras con grietas → faltan **datos** de ese tipo |
| el fondo liso / todo parejo | usa brillo o textura global como atajo |
| la grieta real | mira lo correcto |

In [ ]:
problemas = sorted(Path("data/problemas").glob("*.jp*g")) + sorted(Path("data/problemas").glob("*.png"))
if problemas:
    gradcam.figura_gradcam(m, problemas[:6], arquitectura=ARQ, ruta_salida="results/gradcam_problemas.png")
else:
    print("Poné fotos en data/problemas/ (la pared con la mano, etc.)")

## Paso 5 · Corrección del falso positivo: la puerta "¿esto parece concreto?"
**El problema de fondo:** el clasificador solo conoce *dos* respuestas. Ante algo que nunca vio (una mano, una pared
pintada) no puede decir "esto no es concreto": fuerza una de las dos.

**La corrección (en tres capas, de más a menos importante):**
1. **Datos:** negativos difíciles reales en el entrenamiento (cuaderno 02, paso 2).
2. **Puerta de concreto:** distancia de Mahalanobis sobre el embedding de 1.280 números. Si la imagen queda lejos
   de la nube de concreto, el sistema responde *INDETERMINADO* en vez de afirmar.
3. **Banda de duda:** entre 0,20 y 0,80 de probabilidad el sistema no afirma.

**Qué medir:** el **AUROC** de la puerta (1,0 = separa perfecto · 0,5 = azar) y la **tasa de rechazo de concreto legítimo**
(por diseño ~1 %). No se asume que funciona: se mide con negativos difíciles de **escenas que no vio**.

In [ ]:
emb_modelo = modelo.modelo_embeddings(m, ARQ)
emb = lambda x: emb_modelo.predict(np.asarray(x, dtype=np.float32), batch_size=64, verbose=0)
tr = pd.read_csv("results/particion_entrenamiento.csv")
conc_tr = tr[tr.fuente == "surface_crack"].sample(4000, random_state=1)
conc_va = va[va.fuente == "surface_crack"].sample(1500, random_state=1)
puerta = ood.PuertaConcreto(percentil=99).ajustar(entrenar.predecir_rutas(emb, conc_tr["ruta"].tolist()),
                                                  entrenar.predecir_rutas(emb, conc_va["ruta"].tolist()))
puerta.guardar(f"models/{NOMBRE}/puerta.npz")
print("umbral de distancia:", round(puerta.umbral, 2))

umbral de distancia: 88.25


In [ ]:
# Con los negativos difíciles de escenas NO vistas (los que están en la partición de prueba):
dificiles = te[te.fuente == "negativos_dificiles"]
if len(dificiles):
    e_dif = entrenar.predecir_rutas(emb, dificiles["ruta"].tolist())
    e_con = entrenar.predecir_rutas(emb, te[te.fuente == "surface_crack"].head(1000)["ruta"].tolist())
    print("AUROC de la puerta:", round(puerta.auroc(e_con, e_dif), 3))
    print("concreto legítimo rechazado:", f"{100*(1-puerta.es_concreto(e_con).mean()):.1f} %")
    print("negativos difíciles rechazados (lo que queremos):", f"{100*(1-puerta.es_concreto(e_dif).mean()):.1f} %")
else:
    print("Falta la fuente negativos_dificiles en la partición de prueba (docs/CORRECCION_FALSO_POSITIVO.md).")

## Paso 6 · El sistema completo sobre tus fotos
Encadena: puerta → clasificación (mosaico si la foto es grande) → ancho → inclinación → riesgo. Para el ancho en mm hay
que dar la escala (`mm_por_px`) o las 4 esquinas de un objeto de referencia (`esquinas_ref`, `ref_mm`).

In [11]:
from PIL import Image
sistema = predecir.Sistema(predecir_fn, embeddings=emb, puerta=puerta)
for ruta in problemas[:6]:
    foto = np.asarray(Image.open(ruta).convert("RGB"))
    r = sistema.analizar(foto, roll_deg=0.0, pitch_deg=0.0, escalas=(1.0, 0.5))
    print(f"{ruta.name}: p(grieta)={r['prob_grieta']:.2f} · concreto={r['es_concreto']} · "
          f"riesgo={r['riesgo'].nivel} · {r['riesgo'].razones}")

NameError: name 'problemas' is not defined

## Paso 6b · Fotos propias (lo pide el documento del reto, dentro del 30 % del desempeño)
**Qué hace:** analiza con el sistema completo cada foto de `data/fotos_propias/` y mide el desempeño de dos formas:

* **Forzada:** el sistema siempre responde. Es la medición clásica (exactitud, precisión, recall, F1).
* **Con abstención:** puede responder INDETERMINADO. Se reporta la **cobertura** y, sobre todo, cuántas **grietas reales
  quedaron sin diagnóstico**: un INDETERMINADO sobre una grieta real es un dato de seguridad que no se debe esconder.

**Preparación:** `data/fotos_propias/` con las fotos y `etiquetas.csv` (columnas `archivo,etiqueta`; 1 = con grieta).
Si sacaste la foto con una tarjeta o regla de referencia, anotá su escala en `parametros` (mm por píxel) para obtener el
ancho en mm. El giro del celular (`roll_deg`) lo da el sensor de gravedad; sin él, la inclinación no se usa.

In [12]:
sistema = predecir.Sistema(predecir_fn, embeddings=emb, puerta=puerta)
parametros = {   # opcional, por foto: {"foto03.jpg": {"mm_por_px": 0.21, "roll_deg": 3.0, "pitch_deg": 0.0, "elemento": "columna"}}
}
tabla_fp = fotos_propias.analizar_carpeta(sistema, "data/fotos_propias", etiquetas="data/fotos_propias/etiquetas.csv",
                                          parametros=parametros, salida_csv="results/fotos_propias_analisis.csv",
                                          escalas=(1.0, 0.5))
display(tabla_fp)
met = fotos_propias.metricas_fotos_propias(tabla_fp)
Path("results/metricas_fotos_propias.json").write_text(json.dumps(met, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(met, indent=2, ensure_ascii=False))

Fotos propias: 29 · etiquetadas: 29 (13 con grieta, 16 sin grieta)


,archivo,etiqueta,prob_grieta,es_concreto,riesgo,puntaje,ancho_mm,ancho_px,orientacion,inclinacion_deg,modo,notas
0,foto (1).jpg,1,0.1348,False,INDETERMINADO,NaN,None,None,None,None,mosaico,51 % de la foto no parece concreto.
1,foto (10).jpg,1,0.1457,True,BAJO,0.0,None,None,None,None,mosaico,1 % de la foto no parece concreto.
2,foto (11).jpg,1,0.0000,False,INDETERMINADO,NaN,None,None,None,None,mosaico,54 % de la foto no parece concreto.
3,foto (12).jpg,1,0.0022,True,BAJO,0.0,None,None,None,None,mosaico,4 % de la foto no parece concreto.
4,foto (13).jpg,1,0.0000,True,BAJO,0.0,None,None,None,None,mosaico,
5,foto (14).jpg,0,0.0000,True,BAJO,0.0,None,None,None,None,mosaico,2 % de la foto no parece concreto.
6,foto (15).jpg,0,0.0000,True,BAJO,0.0,None,None,None,None,mosaico,
7,foto (16).jpg,0,0.0000,True,BAJO,0.0,None,None,None,None,mosaico,12 % de la foto no parece concreto.
8,foto (17).jpg,0,0.0001,True,BAJO,0.0,None,None,None,None,mosaico,2 % de la foto no parece concreto.
9,foto (18).jpg,0,0.0021,True,BAJO,0.0,None,None,None,None,mosaico,2 % de la foto no parece concreto.


{
  "n": 29,
  "con_grieta": 13,
  "sin_grieta": 16,
  "forzada": {
    "exactitud": 0.5517241379310345,
    "precision": NaN,
    "recall": 0.0,
    "f1": NaN,
    "tp": 0,
    "tn": 16,
    "fp": 0,
    "fn": 13
  },
  "con_abstencion": {
    "n_respondidas": 25,
    "cobertura": 0.8620689655172413,
    "exactitud_en_respondidas": 0.64,
    "falsos_neg_en_respondidas": 9,
    "falsos_pos_en_respondidas": 0,
    "grietas_reales_sin_diagnostico": 4,
    "sin_grieta_sin_diagnostico": 0
  }
}


## Paso 6c · Paredes VISTAS contra paredes NUEVAS (¿memoriza o no generaliza?)
**La pregunta:** ¿el modelo funciona igual con fotos que **vio al entrenar** (tus escenas de `dificiles`) que con fotos **nuevas** (`fotos_propias`)?
Se separa por etiqueta, porque `dificiles` puede tener fotos con y sin grieta, y se usan las particiones con las que el modelo REALMENTE entrenó.

| Vistas | Nuevas | Lectura |
|---|---|---|
| acierta en las vistas (0 % sin grieta, ~100 % con grieta) | acierta en las nuevas | generaliza bien |
| acierta en las vistas | **falla en las nuevas** | **memoriza** y no generaliza |
| muchos | muchos | no aprendió a distinguir pared lisa de grieta (los datos no alcanzaron) |
| muchos | pocos | raro: revisá que las carpetas no estén mezcladas |

Se usa la **probabilidad cruda del modelo** en las dos columnas (sin la puerta), para que la comparación sea justa.
Con pocas fotos, mirá el intervalo de confianza: un solo error mueve mucho el porcentaje.

In [ ]:
tr_m = pd.read_csv(f"models/{NOMBRE}/particion_entrenamiento.csv")      # lo que ESTE modelo realmente vio al entrenar
print("Lo que el modelo vio al entrenar (fuente x etiqueta):")
print(tr_m.groupby(["fuente", "etiqueta"]).size().unstack(fill_value=0).rename(columns={0: "sin_grieta", 1: "con_grieta"}))
vistas = tr_m[tr_m.fuente == "negativos_dificiles"]
fp_tabla = pd.read_csv("results/fotos_propias_analisis.csv").dropna(subset=["etiqueta"])
conjuntos = [("VISTAS sin grieta (dificiles)", vistas[vistas.etiqueta == 0]["ruta"].tolist(), "deberia dar 0 %"),
             ("VISTAS con grieta (dificiles)", vistas[vistas.etiqueta == 1]["ruta"].tolist(), "deberia dar 100 %"),
             ("NUEVAS sin grieta (fotos_propias)", [f"data/fotos_propias/{a}" for a in fp_tabla[fp_tabla.etiqueta == 0]["archivo"]], "deberia dar 0 %"),
             ("NUEVAS con grieta (fotos_propias)", [f"data/fotos_propias/{a}" for a in fp_tabla[fp_tabla.etiqueta == 1]["archivo"]], "deberia dar 100 %")]
filas = []
for nombre, rutas, ideal in conjuntos:
    if not rutas:
        continue
    p = entrenar.predecir_rutas(predecir_fn, rutas)
    k, n = int((p >= config.UMBRAL_DECISION).sum()), len(p)
    lo, hi = evaluar.intervalo_wilson(k, n)
    filas.append({"grupo": nombre, "fotos": n, "dice_con_grieta": k, "%": round(100 * k / n, 1),
                  "IC95_%": f"{100*lo:.0f}-{100*hi:.0f}", "ideal": ideal})
display(pd.DataFrame(filas))
problemas = sorted(Path("data/problemas").glob("*.jp*g")) + sorted(Path("data/problemas").glob("*.png"))
if problemas:
    pp = predecir_fn(entrenar.cargar_imagenes([str(x) for x in problemas]))
    for x, v in zip(problemas, pp):
        print(f"{x.name}: p(grieta) = {v:.2f}")

Lo que el modelo vio al entrenar (fuente x etiqueta):
etiqueta             sin_grieta  con_grieta
fuente                                     
negativos_dificiles         220         215
surface_crack             13884       14036


,grupo,fotos,dice_con_grieta,%,IC95_%,ideal
0,VISTAS sin grieta (dificiles),220,0,0.0,0-2,deberia dar 0 %
1,VISTAS con grieta (dificiles),215,215,100.0,98-100,deberia dar 100 %
2,NUEVAS sin grieta (fotos_propias),16,0,0.0,0-19,deberia dar 0 %
3,NUEVAS con grieta (fotos_propias),13,2,15.4,4-42,deberia dar 100 %


pared_con_mano.jpg: p(grieta) = 0.02


## Paso 6d · Diagnóstico: ¿falla la RED o falla el SISTEMA?
Si el Paso 6b detecta pocas grietas en tus fotos, hay dos sospechosos distintos y no se arreglan igual:

* **La red** no reconoce grietas de fotos de celular (dominio distinto al de Surface Crack).
* **El sistema** (mosaico, escala o puerta de concreto) arruina una red que sí funcionaría.

Esta celda prueba **cuatro formas** de preguntarle a la misma red, sobre tus fotos con etiqueta:

| Columna | Qué hace |
|---|---|
| `p_entera` | achica TODA la foto a 224×224 y pregunta una vez |
| `p_esc_0.5` | corta la foto en teselas a mitad de tamaño, SIN puerta, y toma la más sospechosa |
| `p_esc_0.25` | lo mismo a un cuarto de tamaño (la grieta se ve más fina) |
| `p_sistema` | lo que dijo el sistema completo en el Paso 6b (con puerta) |

**Cómo leerla:** si alguna columna detecta muchas de tus grietas sin disparar falsos positivos, la red SÍ sirve y el problema
es de escala o del sistema. Si ninguna las detecta, el problema es la red y los datos.

No hace falta repetir el Paso 6b: lee `results/fotos_propias_analisis.csv`. Tarda unos minutos en CPU (corta cada foto en cientos de teselas).

In [ ]:
from PIL import Image
tabla_fp = pd.read_csv("results/fotos_propias_analisis.csv")
sis = tabla_fp.set_index("archivo")
filas = []
for ruta in datos.listar_imagenes("data/fotos_propias"):
    if ruta.name not in sis.index or pd.isna(sis.loc[ruta.name, "etiqueta"]):
        continue
    foto = np.asarray(Image.open(ruta).convert("RGB"))
    entera = np.asarray(Image.fromarray(foto).resize((config.IMG, config.IMG)), dtype=np.float32)[None]
    fila = {"archivo": ruta.name, "etiqueta": int(sis.loc[ruta.name, "etiqueta"]),
            "tamano": f"{foto.shape[1]}x{foto.shape[0]}", "p_entera": round(float(predecir_fn(entera)[0]), 3)}
    for esc in (0.5, 0.25):
        fila[f"p_esc_{esc}"] = round(float(mosaico.clasificar_mosaico(foto, predecir_fn, escalas=(esc,))["prob_max"]), 3)
    fila["p_sistema"] = round(float(sis.loc[ruta.name, "prob_grieta"]), 3)
    filas.append(fila)
diag = pd.DataFrame(filas).sort_values(["etiqueta", "archivo"]).reset_index(drop=True)
display(diag)
print("\nCuántas se detectan con cada método (umbral 0,5):")
for col in ["p_entera", "p_esc_0.5", "p_esc_0.25", "p_sistema"]:
    pos, neg = diag[diag.etiqueta == 1], diag[diag.etiqueta == 0]
    print(f"  {col:11s} grietas detectadas: {(pos[col] >= 0.5).sum():2d}/{len(pos)} · falsos positivos: {(neg[col] >= 0.5).sum():2d}/{len(neg)}")
diag.to_csv("results/diagnostico_fotos_propias.csv", index=False)

,archivo,etiqueta,tamano,p_entera,p_esc_0.5,p_esc_0.25,p_sistema
0,foto (14).jpg,0,960x1280,0.000,0.000,0.000,0.000
1,foto (15).jpg,0,960x1280,0.000,0.000,0.000,0.000
2,foto (16).jpg,0,960x1280,0.000,0.000,0.000,0.000
3,foto (17).jpg,0,960x1280,0.000,0.002,0.000,0.000
4,foto (18).jpg,0,960x1280,0.014,0.005,0.007,0.002
5,foto (19).jpg,0,960x1280,0.000,0.000,0.000,0.000
6,foto (20).jpg,0,960x1280,0.000,0.000,0.000,0.000
7,foto (21).jpg,0,1280x960,0.000,0.000,0.000,0.000
8,foto (22).jpg,0,960x1280,0.000,0.000,0.000,0.000
9,foto (23).jpg,0,960x1280,0.001,0.002,0.000,0.000



Cuántas se detectan con cada método (umbral 0,5):
  p_entera    grietas detectadas:  2/13 · falsos positivos:  0/16
  p_esc_0.5   grietas detectadas:  3/13 · falsos positivos:  0/16
  p_esc_0.25  grietas detectadas:  3/13 · falsos positivos:  0/16
  p_sistema   grietas detectadas:  0/13 · falsos positivos:  0/16


## Paso 7 · Fuera de dominio: SDNET2018 y fotos de campo (solo evaluación)
Estas fuentes **no se usaron para entrenar** (se apartaron en el cuaderno 01, paso 1b). Es la prueba honesta de generalización:
si el recall aquí es mucho menor que en `surface_crack`, el modelo no generaliza y hay que ampliar los datos.

In [ ]:
fuera = pd.read_csv("results/fuera_de_dominio.csv")
for fuente in ("sdnet_fuera", "campo"):
    sub = fuera[fuera.fuente == fuente]
    if len(sub):
        sub = sub.sample(min(1500, len(sub)), random_state=1)
        p = entrenar.predecir_rutas(predecir_fn, sub["ruta"].tolist())
        r = evaluar.metricas(sub["etiqueta"].to_numpy(), p)
        print(f"{fuente}: recall {r['recall']:.3f} · precisión {r['precision']:.3f} · exactitud {r['exactitud']:.3f} · FN {r['fn']} · FP {r['fp']}")
    else:
        print(f"{fuente}: sin imágenes todavía")

## Paso 8 · Tabla comparativa final (para el informe)
Une la línea base (cuaderno 01), MobileNetV2 y EfficientNetB0 en **la misma partición de prueba**. Agregá el tamaño
del `.tflite` y el tiempo de inferencia si vas a justificar la elección del modelo.

In [14]:
filas = []
lb = pd.read_csv("results/linea_base.csv")
for _, f in lb.iterrows():
    filas.append({"modelo": "línea base: " + f["metodo"], "exactitud": f["exactitud"], "recall": f["recall"], "precision": f["precision"], "f1": f["f1"], "FN": f["falsos_neg"], "FP": f["falsos_pos"]})
for nombre, arq in [(n, a) for n, a in (("v2_mobilenet", "mobilenetv2"), ("v2_efficientnet", "efficientnetb0")) if Path(f"models/{n}/modelo.keras").exists()]:
    mm = keras.models.load_model(f"models/{nombre}/modelo.keras")
    p = entrenar.predecir_rutas(entrenar.predictor(modelo.modelo_inferencia(mm, arq)), te["ruta"].tolist())
    r = evaluar.metricas(te["etiqueta"].to_numpy(), p)
    filas.append({"modelo": nombre, "exactitud": round(r["exactitud"], 4), "recall": round(r["recall"], 4), "precision": round(r["precision"], 4), "f1": round(r["f1"], 4), "FN": r["fn"], "FP": r["fp"]})
tabla_final = pd.DataFrame(filas)
display(tabla_final)
tabla_final.to_csv("results/tabla_comparativa.csv", index=False)

,modelo,exactitud,recall,precision,f1,FN,FP
0,línea base: umbral de brillo,0.7027,0.7392,0.6837,0.7104,386,506
1,línea base: umbral de bordes,0.8487,0.9669,0.7794,0.8631,49,405
2,línea base: regresión logística (5 rasgos),0.9827,0.9818,0.9831,0.9824,27,25
3,v2_mobilenet,0.9995,0.9993,0.9997,0.9995,2,1
